In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/capstone-ml'
DATA = f'{ROOT}/data'
PRED = f'{ROOT}/predictions'
MODELS = f'{ROOT}/models'
LOGS = f'{ROOT}/logs'

import sys, importlib.metadata as md
print('Python', sys.version.split()[0])
for p in ['scikit-learn', 'pandas', 'numpy', 'joblib']:
    print(f'{p}=={md.version(p)}')

Mounted at /content/drive
Python 3.13.18
scikit-learn==1.6.1
pandas==2.2.3
numpy==2.4.6
joblib==1.6.0


In [2]:
import json
import pandas as pd

splits = {s: pd.read_csv(f'{DATA}/sst5_{s}.csv') for s in ['train', 'val', 'test']}
for s, df in splits.items():
    print(f'{s}: {len(df)} rows, labels {df["label"].value_counts().sort_index().tolist()}')

with open(f'{DATA}/class_weights.json') as f:
    CLASS_WEIGHTS = {int(k): v for k, v in json.load(f)['weights'].items()}
print('Class weights:', CLASS_WEIGHTS)

assert splits['train']['label'].value_counts().sort_index().tolist() == [3310, 1624, 3610]

train: 8544 rows, labels [3310, 1624, 3610]
val: 1101 rows, labels [428, 229, 444]
test: 2210 rows, labels [912, 389, 909]
Class weights: {0: 0.8604, 1: 1.7537, 2: 0.7889}


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X = {'train': vectorizer.fit_transform(splits['train']['text'])}
X['val'] = vectorizer.transform(splits['val']['text'])
X['test'] = vectorizer.transform(splits['test']['text'])
y = {s: df['label'].values for s, df in splits.items()}

vocab = vectorizer.vocabulary_
n_bigrams = sum(' ' in term for term in vocab)
print(f'Vocabulary: {len(vocab)} terms ({len(vocab) - n_bigrams} single words, {n_bigrams} word pairs)')
for s in ['train', 'val', 'test']:
    print(f'{s}: matrix {X[s].shape}, average {X[s].getnnz(axis=1).mean():.1f} non-zero features per sentence')

empty_val = (X['val'].getnnz(axis=1) == 0).sum()
print(f'Validation sentences with no known terms at all: {empty_val}')

Vocabulary: 21834 terms (8179 single words, 13655 word pairs)
train: matrix (8544, 21834), average 21.7 non-zero features per sentence
val: matrix (1101, 21834), average 19.7 non-zero features per sentence
test: matrix (2210, 21834), average 19.5 non-zero features per sentence
Validation sentences with no known terms at all: 0


In [4]:
Xt = X['train']
mean_sq = Xt.multiply(Xt).mean()
var = mean_sq - Xt.mean() ** 2
print(f"gamma='scale' = 1 / (n_features x variance) = {1 / (Xt.shape[1] * var):.3f}")

gamma='scale' = 1 / (n_features x variance) = 1.002


In [5]:
import time
from sklearn.svm import SVC
from sklearn.metrics import f1_score, accuracy_score

grid = []
for C in [1, 10, 100]:
    for gamma in ['scale', 0.1, 1]:
        start = time.perf_counter()
        svm = SVC(kernel='rbf', C=C, gamma=gamma, class_weight=CLASS_WEIGHTS)
        svm.fit(X['train'], y['train'])
        pred = svm.predict(X['val'])
        grid.append({'C': C, 'gamma': gamma,
                     'val_macro_f1': round(f1_score(y['val'], pred, average='macro'), 4),
                     'val_accuracy': round(accuracy_score(y['val'], pred), 4),
                     'support_vectors': int(svm.n_support_.sum()),
                     'fit_seconds': round(time.perf_counter() - start, 1)})
        print(grid[-1])

grid_df = pd.DataFrame(grid).sort_values('val_macro_f1', ascending=False)
grid_df.to_csv(f'{LOGS}/svm_grid.csv', index=False)
print()
print(grid_df.to_string(index=False))

{'C': 1, 'gamma': 'scale', 'val_macro_f1': 0.5416, 'val_accuracy': 0.6349, 'support_vectors': 8481, 'fit_seconds': 36.3}
{'C': 1, 'gamma': 0.1, 'val_macro_f1': 0.5424, 'val_accuracy': 0.5777, 'support_vectors': 8468, 'fit_seconds': 36.1}
{'C': 1, 'gamma': 1, 'val_macro_f1': 0.5436, 'val_accuracy': 0.6358, 'support_vectors': 8481, 'fit_seconds': 34.8}
{'C': 10, 'gamma': 'scale', 'val_macro_f1': 0.5165, 'val_accuracy': 0.6349, 'support_vectors': 8472, 'fit_seconds': 35.2}
{'C': 10, 'gamma': 0.1, 'val_macro_f1': 0.5297, 'val_accuracy': 0.5949, 'support_vectors': 7809, 'fit_seconds': 33.7}
{'C': 10, 'gamma': 1, 'val_macro_f1': 0.5165, 'val_accuracy': 0.6349, 'support_vectors': 8472, 'fit_seconds': 35.6}
{'C': 100, 'gamma': 'scale', 'val_macro_f1': 0.5165, 'val_accuracy': 0.6349, 'support_vectors': 8471, 'fit_seconds': 34.8}
{'C': 100, 'gamma': 0.1, 'val_macro_f1': 0.519, 'val_accuracy': 0.5895, 'support_vectors': 7708, 'fit_seconds': 35.1}
{'C': 100, 'gamma': 1, 'val_macro_f1': 0.5165, 'va

In [6]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

BEST = {'C': 1, 'gamma': 1}

svm = SVC(kernel='rbf', C=BEST['C'], gamma=BEST['gamma'], class_weight=CLASS_WEIGHTS,
          probability=True, random_state=42)
start = time.perf_counter()
svm.fit(X['train'], y['train'])
print(f'Fitted in {time.perf_counter() - start:.0f} seconds')

val_pred = svm.predict(X['val'])
val_proba = svm.predict_proba(X['val'])

print('\nValidation, per class:')
print(classification_report(y['val'], val_pred, target_names=['negative', 'neutral', 'positive'], digits=4))
print('Confusion matrix (rows = true, columns = predicted):')
print(pd.DataFrame(confusion_matrix(y['val'], val_pred),
                   index=['true neg', 'true neu', 'true pos'], columns=['pred neg', 'pred neu', 'pred pos']))

disagree = (val_proba.argmax(axis=1) != val_pred).sum()
print(f'\nSentences where the top probability disagrees with the SVM decision: {disagree} of {len(val_pred)}')

Fitted in 166 seconds

Validation, per class:
              precision    recall  f1-score   support

    negative     0.6472    0.7243    0.6836       428
     neutral     0.3646    0.1528    0.2154       229
    positive     0.6749    0.7995    0.7320       444

    accuracy                         0.6358      1101
   macro avg     0.5622    0.5589    0.5436      1101
weighted avg     0.5996    0.6358    0.6057      1101

Confusion matrix (rows = true, columns = predicted):
          pred neg  pred neu  pred pos
true neg       310        39        79
true neu       102        35        92
true pos        67        22       355

Sentences where the top probability disagrees with the SVM decision: 44 of 1101


In [8]:
import os
import joblib
from datetime import date

def prediction_frame(df, pred, proba, eval_set):
    return pd.DataFrame({
        'id': df['id'], 'true_label': df['label'], 'pred_label': pred,
        'p_neg': proba[:, 0].round(6), 'p_neu': proba[:, 1].round(6), 'p_pos': proba[:, 2].round(6),
        'model': 'svm', 'seed': '', 'train_data': 'sst5', 'eval_set': eval_set,
    })

test_pred = svm.predict(X['test'])
test_proba = svm.predict_proba(X['test'])

prediction_frame(splits['val'], val_pred, val_proba, 'sst5_val').to_csv(f'{PRED}/svm_sst5_val.csv', index=False)
prediction_frame(splits['test'], test_pred, test_proba, 'sst5_test').to_csv(f'{PRED}/svm_sst5_test.csv', index=False)

val_f1 = f1_score(y['val'], val_pred, average='macro')
test_f1 = f1_score(y['test'], test_pred, average='macro')
print(f'SST-5 val:  macro-F1 {val_f1:.4f}, accuracy {accuracy_score(y["val"], val_pred):.4f}')
print(f'SST-5 test: macro-F1 {test_f1:.4f}, accuracy {accuracy_score(y["test"], test_pred):.4f}')

pipeline = Pipeline([('tfidf', vectorizer), ('svm', svm)])
os.makedirs(MODELS, exist_ok=True)
joblib.dump(pipeline, f'{MODELS}/svm.joblib')
print(f'Saved {MODELS}/svm.joblib ({os.path.getsize(f"{MODELS}/svm.joblib") / 1e6:.1f} MB)')

log = f'{LOGS}/experiments.csv'
run_id = 'svm-sst5'
logged = pd.read_csv(log)
if run_id not in logged['run_id'].astype(str).values:
    pd.DataFrame([{'run_id': run_id, 'model': 'svm', 'seed': '', 'train_data': 'sst5',
                   'hyperparameters': 'tfidf 1-2gram sublinear min_df=2; rbf C=1 gamma=1; class weights from json',
                   'val_macro_f1': round(val_f1, 4), 'test_macro_f1': round(test_f1, 4),
                   'date': date.today().isoformat()}]).to_csv(log, mode='a', header=False, index=False)
    print('Logged', run_id)
else:
    print(run_id, 'already in the experiment log, not added again')

SST-5 val:  macro-F1 0.5436, accuracy 0.6358
SST-5 test: macro-F1 0.5399, accuracy 0.6462
Saved /content/drive/MyDrive/capstone-ml/models/svm.joblib (3.1 MB)
Logged svm-sst5
